# Step Functions pipeline

Deploys the workflow `pf-batch-pipeline` and runs it. One execution processes **one batch** from start to finish:

```
Ingest (Lambda)  ->  BronzeToSilver (Glue job)  ->  MergeDimPatient -> MergeDimFacility -> MergeDimPractitioner -> MergeFactEncounter (Athena)
```
Start it with `{"batch": N}`. If any step fails, the workflow stops and reports the error.
Run the cells top to bottom. A full run takes about 4 minutes (the Glue job needs about 2).

## 1. Setup

In [1]:
import json
import sys
import time
from datetime import datetime
from pathlib import Path

import boto3

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

# deploy.py is the same deployment code GitHub Actions uses; aws_json and run_athena call AWS (see infra/)
import deploy
from aws_helpers import aws_json, run_athena, whoami

CONFIG = deploy.load_config()
SESSION = boto3.Session(region_name=CONFIG["region"])
NAME = CONFIG["state_machine"]

print("identity     :", whoami())
print("state machine:", NAME)

identity     : user/Ahmed
state machine: pf-batch-pipeline


## 2. Deploy the workflow
`deploy_state_machine` builds the definition from `step_functions/pf_batch_pipeline.asl.json`, puts the text of the four
`MERGE` files into it, creates the CloudWatch log group (errors only, kept 30 days) and creates or updates the state machine.

In [2]:
STATE_MACHINE_ARN = deploy.deploy_state_machine(SESSION, CONFIG)
print("ARN ends with:", STATE_MACHINE_ARN.split(":", 5)[-1])

created state machine pf-batch-pipeline
ARN ends with: stateMachine:pf-batch-pipeline


In [3]:
# AWS checks the definition itself (states, transitions, intrinsic functions)
definition = deploy.state_machine_definition(CONFIG, boto3.client("sts").get_caller_identity()["Account"])
check = boto3.client("stepfunctions").validate_state_machine_definition(definition=json.dumps(definition), type="STANDARD")
assert check["result"] == "OK", check["diagnostics"]

machine = aws_json("stepfunctions", "describe-state-machine", "--state-machine-arn", STATE_MACHINE_ARN)
assert machine["status"] == "ACTIVE" and machine["type"] == "STANDARD"
assert machine["loggingConfiguration"]["level"] == "ERROR"
print(f"OK  {NAME}: {machine['status']}, {machine['type']}, logging level {machine['loggingConfiguration']['level']}")
print("steps:", " -> ".join(definition["States"].keys()))

OK  pf-batch-pipeline: ACTIVE, STANDARD, logging level ERROR
steps: Ingest -> BronzeToSilver -> PrepareDateParameter -> MergeDimPatient -> MergeDimFacility -> MergeDimPractitioner -> MergeFactEncounter -> PipelineSucceeded -> PipelineFailed


## 3. Helpers: run the pipeline and read its history
`run_pipeline` starts an execution and waits for it to finish. `step_times` reads the execution history and returns
how long each step took (the history is what the Step Functions console draws as the colored picture).

In [4]:
def run_pipeline(batch, wait=True):
    started = aws_json("stepfunctions", "start-execution", "--state-machine-arn", STATE_MACHINE_ARN,
                       "--name", f"batch-{batch}-{int(time.time())}", "--input", json.dumps({"batch": batch}))
    arn = started["executionArn"]
    return wait_for(arn) if wait else arn


def wait_for(execution_arn, timeout=1500):
    deadline = time.time() + timeout
    while True:
        execution = aws_json("stepfunctions", "describe-execution", "--execution-arn", execution_arn)
        if execution["status"] != "RUNNING":
            return execution
        assert time.time() < deadline, "execution still running"
        time.sleep(10)


def history(execution_arn):
    return aws_json("stepfunctions", "get-execution-history", "--execution-arn", execution_arn)["events"]


def step_times(execution_arn):
    entered, rows = {}, []
    for event in history(execution_arn):
        kind = event["type"]
        if kind.endswith("StateEntered"):
            entered[event["stateEnteredEventDetails"]["name"]] = event["timestamp"]
        elif kind.endswith("StateExited"):
            name = event["stateExitedEventDetails"]["name"]
            seconds = (datetime.fromisoformat(event["timestamp"]) - datetime.fromisoformat(entered[name])).total_seconds()
            rows.append((name, round(seconds)))
    return rows


def show_steps(execution_arn):
    for name, seconds in step_times(execution_arn):
        print(f"  {name:<24}{seconds:>5} s")

## 4. Run batch 1
The gold tables already contain batch 1, so this run also proves the whole pipeline can be repeated without creating duplicates.

In [5]:
gold_before = {t: int(run_athena(f"SELECT COUNT(*) FROM pf_gold.{t}")["rows"][0][0])
               for t in ("dim_patient", "dim_facility", "dim_practitioner", "fact_encounter")}

run = run_pipeline(1)
print("status  :", run["status"])
print("output  :", run["output"])
show_steps(run["executionArn"])
assert run["status"] == "SUCCEEDED", (run.get("error"), run.get("cause"))
assert json.loads(run["output"]) == {"batch": 1, "ingest_date": "2026-11-01", "file_count": 37}

status  : SUCCEEDED
output  : {"file_count":37,"batch":1,"ingest_date":"2026-11-01"}


  Ingest                     10 s
  BronzeToSilver            174 s
  PrepareDateParameter        0 s
  MergeDimPatient            61 s
  MergeDimFacility           62 s
  MergeDimPractitioner       58 s
  MergeFactEncounter         56 s
  PipelineSucceeded           0 s


In [6]:
gold_after = {t: int(run_athena(f"SELECT COUNT(*) FROM pf_gold.{t}")["rows"][0][0]) for t in gold_before}
assert gold_after == gold_before, (gold_before, gold_after)
print("OK  gold row counts unchanged after the rerun:", gold_after)

OK  gold row counts unchanged after the rerun: {'dim_patient': 137, 'dim_facility': 521, 'dim_practitioner': 521, 'fact_encounter': 4586}


## 5. A bad batch must fail clearly
Batch 5 does not exist (the project has 4). The Lambda refuses it, so the workflow must stop at the first step with the
Lambda's message, and the Glue job and the merges must not run.

In [7]:
bad = run_pipeline(5)
print("status:", bad["status"])
print("error :", bad["error"])
print("cause :", json.loads(bad["cause"])["errorMessage"])
assert bad["status"] == "FAILED"
assert "between 1 and 4" in bad["cause"]

visited = [name for name, _ in step_times(bad["executionArn"])]
print("steps that ran:", visited)
assert "BronzeToSilver" not in visited and not any(step.startswith("Merge") for step in visited)
print("OK  the pipeline stopped at the first step; nothing else ran")

status: FAILED
error : ValueError
cause : batch must be between 1 and 4, got 5


steps that ran: ['Ingest']
OK  the pipeline stopped at the first step; nothing else ran


## 6. Two batches at the same moment
The Glue job allows one run at a time. Start batches 3 and 4 together: the second must **wait and retry** at the Glue step instead of failing,
and both must finish successfully.

In [8]:
first = run_pipeline(3, wait=False)
second = run_pipeline(4, wait=False)
runs = {3: wait_for(first), 4: wait_for(second)}

for batch, execution in runs.items():
    print(f"batch {batch}: {execution['status']}  {execution.get('output', execution.get('cause', ''))[:90]}")
    show_steps(execution["executionArn"])

retries = 0
for execution in runs.values():
    retries += sum(1 for e in history(execution["executionArn"])
                   if e["type"] == "TaskFailed" and "ConcurrentRunsExceeded" in e["taskFailedEventDetails"].get("error", ""))
print("\nGlue 'job is busy' answers that were retried:", retries)
assert all(execution["status"] == "SUCCEEDED" for execution in runs.values()), runs

batch 3: SUCCEEDED  {"file_count":34,"batch":3,"ingest_date":"2026-11-03"}


  Ingest                      8 s
  BronzeToSilver            124 s
  PrepareDateParameter        0 s
  MergeDimPatient            65 s
  MergeDimFacility           65 s
  MergeDimPractitioner       63 s
  MergeFactEncounter         62 s
  PipelineSucceeded           0 s
batch 4: SUCCEEDED  {"file_count":34,"batch":4,"ingest_date":"2026-11-04"}


  Ingest                      9 s
  BronzeToSilver            271 s
  PrepareDateParameter        0 s
  MergeDimPatient            65 s
  MergeDimFacility           63 s
  MergeDimPractitioner       57 s
  MergeFactEncounter         62 s
  PipelineSucceeded           0 s



Glue 'job is busy' answers that were retried: 3


## 7. The gold layer after all of this

In [9]:
final = {t: int(run_athena(f"SELECT COUNT(*) FROM pf_gold.{t}")["rows"][0][0]) for t in gold_before}
assert final == gold_before, (gold_before, final)
for table, key in (("dim_patient", "patient_id"), ("fact_encounter", "encounter_id"), ("dim_practitioner", "npi")):
    duplicates = int(run_athena(f"SELECT COUNT(*) - COUNT(DISTINCT {key}) FROM pf_gold.{table}")["rows"][0][0])
    assert duplicates == 0, table
print("OK  row counts unchanged and no duplicate keys:", final)

OK  row counts unchanged and no duplicate keys: {'dim_patient': 137, 'dim_facility': 521, 'dim_practitioner': 521, 'fact_encounter': 4586}


## 8. Where to look in the AWS console

In [10]:
executions = aws_json("stepfunctions", "list-executions", "--state-machine-arn", STATE_MACHINE_ARN, "--max-results", "6")["executions"]
for e in executions:
    print(f"{e['startDate'][11:19]}  {e['status']:<10}  {e['name']}")
print("\nStep Functions console > State machines >", NAME, "> Executions: click one to see the colored picture of the steps.")
print("Errors are also written to the CloudWatch log group /aws/vendedlogs/states/" + NAME)

22:26:49  SUCCEEDED   batch-4-1793035607
22:26:47  SUCCEEDED   batch-3-1793035605
22:26:42  FAILED      batch-5-1793035600
22:19:05  SUCCEEDED   batch-1-1793035143

Step Functions console > State machines > pf-batch-pipeline > Executions: click one to see the colored picture of the steps.
Errors are also written to the CloudWatch log group /aws/vendedlogs/states/pf-batch-pipeline


## What this workflow is for

| Piece | Role |
|---|---|
| `step_functions/pf_batch_pipeline.asl.json` | The workflow definition (the SQL of the merges is inserted at deploy time) |
| `infra/deploy.py` | Builds and deploys it; GitHub Actions runs the same code after a push |
| Execution input `{"batch": N}` | The only thing needed to process a batch |

Retries are built in: the Lambda and Athena are retried on temporary errors, and the Glue step waits for a busy job.
Every failure ends in the `PipelineFailed` state with the original error and cause.

**Next:** the folder trigger. A marker file `landing/batch_NN/_SUCCESS` will start this workflow automatically.